[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jac99/pw-llm-lab/blob/main/labs/lab3/lab3_finetuning_student.ipynb)

# Lab 3 – Dostosowywanie generatywnych modeli językowych

**Cele:**
- stosować in-context learning (zero-shot i few-shot) z modelem generatywnym
- dostrajać model metodami PEFT: LoRA, prompt tuning, p-tuning
- porównać jakość i koszt metod dostrajania

**Wymagania:** GPU T4 (*Środowisko wykonawcze → Zmień typ środowiska wykonawczego → T4 GPU*).  
**Szacowany czas:** ~2–3 h, w tym ~30 min treningu na GPU  
**Ocena:** wykonanie zadań 1–2 (zadanie 3 opcjonalne); kryteria zaliczenia w opisie zadania poniżej

Zacznij od uruchomienia komórki *Konfiguracja środowiska* poniżej.

In [ ]:
# @title Konfiguracja środowiska (uruchom jako pierwszą)
%pip install -q "transformers==4.57.3" "peft==0.17.1" "bitsandbytes==0.48.2" "datasets==4.4.1" "accelerate==1.11.0" "colorama==0.4.6"
import os
import random

import numpy as np
import torch

# Tryb skrócony (mniej danych i kroków), używany m.in. w testach automatycznych.
FAST_MODE = os.environ.get("FAST_MODE", "0") == "1"

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    assert FAST_MODE, "Brak GPU. Włącz: Środowisko wykonawcze → Zmień typ środowiska wykonawczego → T4 GPU"
print("PyTorch", torch.__version__, "| FAST_MODE:", FAST_MODE)

## Opis zadania do wykonania

Celem zadania jest **dostosowanie modelu Phi-1.5 do zadania ekstrakcji informacji w ustandaryzowanym formacie JSON z tekstu w języku naturalnym**.

Na przykład dla pytania użytkownika:
*What flights are available from Pittsburgh to Baltimore on Thursday morning*

chcemy wygenerować ustrukturyzowaną reprezentację w formacie JSON:
```
{
    "depart_date.day_name": "thursday",
    "depart_time.period_of_day": "morning",
    "fromloc.city_name": "pittsburgh",
    "toloc.city_name": "baltimore"
}
```

W notatniku należy **zaimplementować następujące metody**:

1.   Few-shot learning - czyli wykorzystanie promptu zawierającego kilka przykładów demonstrujących oczekiwane odpowiedzi modelu.

2.   Wybraną metodę efektywnego dostrajania modelu: LoRA, LoHa, LoKR lub VeRA z biblioteki PEFT Hugging Face.

3.   (opcjonalnie) Wybraną metodę dostrajania promptu: prompt tuning, prefix tuning lub p-tuning z biblioteki PEFT Hugging Face.

Rozwiązując zadanie, skorzystaj z implementacji w notatniku *Dostosowywanie generatywnych modeli językowych z wykorzystaniem biblioteki Hugging Face – notatnik wykładowy* ([`lab3_finetuning_lecture.ipynb`](lab3_finetuning_lecture.ipynb)). Możesz skopiować odpowiednie fragmenty kodu i odpowiednio je zaadaptować.

**Uwaga:**
* Aby zaliczyć laboratorium, nie jest wymagane, aby dostrojony model generował odpowiedzi zawierające wszystkie oczekiwane atrybuty, ani aby wszystkie ich wartości były poprawne.
Ważne, aby odpowiedź była w oczekiwanym formacie JSON.
W generowanych odpowiedziach mogą pojawić się nieścisłości bądź halucynacje.
* Przykłady treningowe przygotowujemy tak, aby odpowiedź kończyła się tokenem końca tekstu (EOS), a funkcja straty była liczona tylko na odpowiedzi (funkcja `tokenize_completion_only`). Dzięki temu dostrojony model powinien kończyć generowanie po symbolu `}`. Jeśli po `}` pojawi się dalszy tekst, zostanie on pominięty przy ocenie.
* Odpowiedzi modeli oceń ilościowo funkcją `evaluate_json` (odsetek poprawnych odpowiedzi JSON oraz precyzja, czułość i F1 na poziomie slotów) i porównaj metody w tabeli na końcu notatnika.


# Przygotowanie środowiska

In [ ]:
!nvidia-smi

Pakiety wykorzystywane w notatniku są instalowane (w przypiętych wersjach) w komórce *Konfiguracja środowiska* na początku notatnika.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# Przygotowanie zbioru danych

W notatniku zostanie wykorzystany zbiór danych **ATIS** (*Airline Travel Information System*) zawierający zapytania dotyczące podróży lotniczych w języku naturalnym.
Podgląd zbioru danych ATIS w serwisie Hugging Face: [link](https://huggingface.co/datasets/tuetschek/atis).

Zbiór walidacyjny (do oceny modelu w trakcie treningu) wydzielimy ze zbioru treningowego. Zbiór testowy posłuży wyłącznie do końcowej oceny.

In [ ]:
from datasets import load_dataset

train_dataset = load_dataset("tuetschek/atis", split='train')
test_dataset = load_dataset("tuetschek/atis", split='test')

# Ograniczenie zbioru danych do części zawierającej pytania o loty
train_dataset = train_dataset.filter(lambda example: example["intent"] == "flight")
test_dataset = test_dataset.filter(lambda example: example["intent"] == "flight")

# W FAST_MODE (testy na CPU) korzystamy z małych wycinków zbiorów
if FAST_MODE:
    train_dataset = train_dataset.select(range(80))
    test_dataset = test_dataset.select(range(8))

# Wydzielenie zbioru walidacyjnego ze zbioru treningowego
split = train_dataset.train_test_split(test_size=16 if FAST_MODE else 200, seed=SEED)
train_dataset, val_dataset = split["train"], split["test"]

In [ ]:
print("Zbiór treningowy")
print(train_dataset)
print("Zbiór walidacyjny")
print(val_dataset)
print("Zbiór testowy")
print(test_dataset)

Przykładowy element zbioru danych:

In [ ]:
i = 15
dataset_item = train_dataset[i]

for key in train_dataset[i]:
  print(f"{key}: {dataset_item[key]}")

Element zbioru danych jest słownikiem.
W zadaniu wykorzystamy następujące pola:

*   **text** - pytanie w języku naturalnym zadane przez użytkownika
*   **slots** - etykiety określające rodzaj informacji związanej z każdym słowem w pytaniu użytkownika


Etykiety w polu `slots` wykorzystamy do utworzenia oczekiwanego wyjścia modelu językowego w formacie JSON. Etykiety oznaczają:
*   `O` (*outside*) - Informacja (słowo) do pominięcia
*   `B-{entity}` (*beginning*) - Pierwsze słowo opisujące dany typ informacji (np. `B-class_type` dla słowa "first")
*   `I-{entity}` (*inside*) - Kolejne słowo opisujące dany typ informacji (np. `I-class_type` dla słowa "class")

Sprawdźmy etykiety dla każdego słowa w przykładowym tekście.

In [ ]:
print()
for word, slot in zip(dataset_item["text"].split(), dataset_item["slots"].split()):
  print(f"{word} - {slot}")

## Wstępne przetwarzanie zbioru danych

Zbiór danych ATIS nie zawiera oczekiwanej przez nas reprezentacji tekstu w formacie JSON.

Poniższa funkcja pomocnicza generuje ustrukturyzowaną reprezentację zapytania w języku naturalnym w formacie JSON w oparciu o zawartość pól `text` i `slots`. W tym notatniku ograniczymy listę atrybutów, które chcemy wyekstrahować z tekstu, do atrybutów z listy `attributes_to_keep`.

In [ ]:
import json

# Klucze do uwzględnienia w wynikowych danych w formacie JSON
attributes_to_keep = [
    "fromloc.city_name",
    "toloc.city_name",
    "depart_date.day_name",
    'depart_time.period_of_day',
    "depart_date.day_number",
    "depart_date.month_name",
    "depart_time.time",
    "depart_date.relative",
    'airline_name'
    ]


def convert_to_structured(dataset_item):
    words = dataset_item['text'].split()
    slot_labels = dataset_item['slots'].split()
    assert len(words) == len(slot_labels)

    structured_data = {}
    current_key = None

    for word, label in zip(words, slot_labels):
        key = label[2:]

        # Słowa z etykietą "O" (Outside of entities) i atrybuty spoza listy kończą bieżącą encję
        if label == "O" or key not in attributes_to_keep:
            current_key = None
            continue

        if label.startswith("B-"):       # Początek encji
            current_key = key
            structured_data[current_key] = word
        elif label.startswith("I-") and current_key == key:  # Kontynuacja tej samej encji
            structured_data[current_key] += " " + word

    # Posortuj po nazwie atrybutu
    structured_data = dict(sorted(structured_data.items()))
    return json.dumps(structured_data, indent=4)

Wyświetlenie elementu zbioru danych i wyniku w formacie JSON.

In [ ]:
e = train_dataset[1]
print(e)
structured_output = convert_to_structured(e)
print("\nWynik w formacie JSON")
print(structured_output)

Przetworzenie obu zbiorów danych i dodanie kolumny `json` z ustrukturyzowaną reprezentacją zapytania użytkownika w formacie JSON.

In [ ]:
def add_json_representation(dataset_item):
    dataset_item["json"] = convert_to_structured(dataset_item)
    return dataset_item

train_dataset = train_dataset.map(add_json_representation)
val_dataset = val_dataset.map(add_json_representation)
test_dataset = test_dataset.map(add_json_representation)

In [ ]:
e = train_dataset[1]
for key in e:
  print(f"{key}: {e[key]}")


**Zadanie**

Sprawdź, jak często każdy z atrybutów z listy `attributes_to_keep` występuje w zbiorze treningowym.

## Przygotowanie tokenizatora

In [ ]:
# Na CPU (FAST_MODE, testy automatyczne) zamiast Phi-1.5 używamy małego, losowo zainicjalizowanego
# modelu o tej samej architekturze (PhiForCausalLM, te same nazwy warstw).
base_model_id = "hf-internal-testing/tiny-random-PhiForCausalLM" if FAST_MODE else "microsoft/phi-1_5"

tokenizer = AutoTokenizer.from_pretrained(base_model_id)
tokenizer.pad_token = tokenizer.eos_token
print(f"Rozmiar słownika: {tokenizer.vocab_size}")

Sprawdzenie tokenizatora

In [ ]:
s = "A dog is running very quickly."
tokenized_s = tokenizer(s)
print(tokenized_s)

# Przygotowanie modelu językowego

## Utworzenie instancji pretrenowanego modelu językowego Phi-1.5

Instancję pretrenowanego modelu Phi-1.5 tworzymy funkcją `AutoModelForCausalLM.from_pretrained()`. Na GPU wagi modelu są kwantyzowane do 4 bitów (format NF4, jak w metodzie QLoRA), co zmniejsza zużycie pamięci. W trybie `FAST_MODE` (testy automatyczne na CPU) wczytywany jest mały, losowo zainicjalizowany model o tej samej architekturze, bez kwantyzacji – jego odpowiedzi są przypadkowe.

Funkcja `load_base_model` przyda się również później – do ponownego wczytania modelu bazowego.

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"


def load_base_model(quantize: bool = True):
    """Wczytuje model bazowy. Na GPU (quantize=True) w kwantyzacji 4-bitowej NF4, tak jak w metodzie QLoRA.
    Na CPU (FAST_MODE) bez kwantyzacji – bitsandbytes wymaga GPU."""
    if quantize and torch.cuda.is_available():
        q_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",             # 4-bitowy typ NormalFloat (NF4)
            bnb_4bit_compute_dtype=torch.float16,  # obliczenia w fp16 (T4 nie obsługuje bf16)
        )
        return AutoModelForCausalLM.from_pretrained(base_model_id, quantization_config=q_config,
                                                    dtype=torch.float16, device_map={"": torch.cuda.current_device()})  # indeks GPU, nie "cuda" – wymaga tego accelerate przy treningu modelu 4-bit
    dtype = torch.float16 if torch.cuda.is_available() else torch.float32
    return AutoModelForCausalLM.from_pretrained(base_model_id, dtype=dtype).to(device)


model = load_base_model()

In [ ]:
print(model)

## Funkcje pomocnicze

Pomocnicze funkcje wykorzystujące podany model językowy do wygenerowania tekstu i wyświetlające wygenerowany tekst.
Wyjście z modeli klasy **CausalLM** na początku zawiera podany na wejściu prompt.

In [ ]:
from colorama import Fore
from transformers import BatchEncoding


def generate_text(model: nn.Module, model_input: BatchEncoding, max_new_tokens: int = 100,
                  return_full_text: bool = False) -> str:
  # Generowanie tekstu przez model

  model.eval()
  with torch.no_grad():
    generated_tokens = model.generate(
        input_ids = model_input['input_ids'],
        attention_mask = model_input['attention_mask'],
        max_new_tokens=max_new_tokens,
        num_beams=1,
        do_sample=False,
        eos_token_id=tokenizer.eos_token_id,  # zakończ generowanie po wygenerowaniu tokenu EOS
        pad_token_id=tokenizer.pad_token_id,
        )[0]
    # generated_tokens zawiera tokeny wejściowe i nowo wygenerowane
    if not return_full_text:
      # Tylko nowo wygenerowane tokeny
      generated_tokens = generated_tokens[model_input['input_ids'].shape[1]:]
    return tokenizer.decode(generated_tokens, skip_special_tokens=True)


def generate_and_print_text(model: nn.Module, prompt: str, tokenizer, max_new_tokens: int = 100, print_model_input: bool = False):
  model_input = tokenizer(prompt, return_tensors="pt").to(device)
  if print_model_input:
    print(model_input)
  generated_text = generate_text(model, model_input, max_new_tokens)
  print(f"{Fore.RESET}{prompt}", end="")
  print(f"{Fore.BLUE}{generated_text}{Fore.RESET}")

Funkcja pomocnicza wykorzystywana do tokenizacji przykładów treningowych. Aby podczas trenowania modelu można było utworzyć wsady złożone z kilku elementów, konieczne jest wyrównanie długości tekstów po tokenizacji – każdy przykład jest uzupełniany (*padding*) albo obcinany do długości `max_length`.

Funkcja `tokenize_completion_only` tokenizuje osobno prompt (polecenie i pytanie) oraz oczekiwaną odpowiedź:
* do odpowiedzi dodaje token końca tekstu (EOS), dzięki czemu model uczy się kończyć generowanie po odpowiedzi,
* etykiety (`labels`) tokenów promptu i wypełnienia ustawia na -100, więc funkcja straty jest liczona tylko na odpowiedzi.

Tak przygotowane przykłady nie wymagają dodatkowego przetwarzania – w klasie `Trainer` wystarczy `data_collator=transformers.default_data_collator`. (Kolektor `DataCollatorForLanguageModeling` nadpisałby etykiety i zamaskował również token EOS, bo jest on jednocześnie tokenem wypełnienia.)

In [ ]:
def tokenize_completion_only(prompt: str, answer: str, max_length: int) -> dict[str, list[int]]:
    """Tokenizuje prompt i odpowiedź do stałej długości `max_length`.

    - Na końcu odpowiedzi dodawany jest token EOS, dzięki czemu model uczy się kończyć generowanie.
    - Etykiety (`labels`) promptu i wypełnienia (padding) mają wartość -100, więc funkcja straty
      jest liczona tylko na tokenach odpowiedzi (przesunięcie o jeden token wykonuje sam model).
    """
    prompt_ids = tokenizer(prompt)["input_ids"]
    answer_ids = tokenizer(answer + tokenizer.eos_token)["input_ids"]
    input_ids = (prompt_ids + answer_ids)[:max_length]
    labels = ([-100] * len(prompt_ids) + answer_ids)[:max_length]
    n_pad = max_length - len(input_ids)
    return {
        "input_ids": input_ids + [tokenizer.pad_token_id] * n_pad,
        "attention_mask": [1] * len(input_ids) + [0] * n_pad,
        "labels": labels + [-100] * n_pad,
    }

Funkcje pomocnicze do **ilościowej oceny** odpowiedzi modelu na zbiorze testowym:
* `parse_json` – wyszukuje w wygenerowanym tekście pierwszy obiekt JSON (tekst wygenerowany po `}` jest pomijany),
* `evaluate_json` – dla `n` pierwszych przykładów zbioru generuje odpowiedź (prompt tworzy funkcja `prompt_fn`) i liczy odsetek odpowiedzi będących poprawnym JSON-em (`valid_json`) oraz precyzję, czułość i miarę F1 na poziomie slotów, czyli par klucz–wartość (wielkość liter w wartościach nie ma znaczenia).

Wyniki kolejnych metod zapisuj w słowniku `results` – na końcu notatnika zostaną zebrane w tabeli porównawczej.

In [ ]:
import re


def parse_json(text: str) -> dict | None:
    """Zwraca pierwszy obiekt JSON (słownik) znaleziony w tekście albo None."""
    match = re.search(r"\{.*?\}", text, re.S)
    if match is None:
        return None
    try:
        parsed = json.loads(match.group(0))
    except json.JSONDecodeError:
        return None
    return parsed if isinstance(parsed, dict) else None


def evaluate_json(model: nn.Module, dataset, prompt_fn, n: int | None = None, max_new_tokens: int = 128) -> dict:
    """Odsetek poprawnych odpowiedzi JSON oraz precyzja, czułość i F1 na poziomie slotów (par klucz–wartość)."""
    n = len(dataset) if n is None else min(n, len(dataset))
    n_valid = tp = fp = fn = 0
    for example in dataset.select(range(n)):
        gold = {(key, str(value).lower().strip()) for key, value in json.loads(example["json"]).items()}
        model_input = tokenizer(prompt_fn(example), return_tensors="pt").to(device)
        pred = parse_json(generate_text(model, model_input, max_new_tokens))
        if pred is None:
            fn += len(gold)
            continue
        n_valid += 1
        pred = {(key, str(value).lower().strip()) for key, value in pred.items()}
        tp += len(pred & gold)
        fp += len(pred - gold)
        fn += len(gold - pred)
    precision = tp / max(tp + fp, 1)
    recall = tp / max(tp + fn, 1)
    f1 = 2 * precision * recall / max(precision + recall, 1e-9)
    return {"valid_json": n_valid / n, "precision": precision, "recall": recall, "f1": f1, "n": n}


N_EVAL = len(test_dataset) if FAST_MODE else 50  # liczba przykładów testowych do oceny (pełny zbiór: dłużej)
results = {}  # wyniki metod do tabeli porównawczej

Sprawdzenie działania modelu dla przykładowego promptu.

In [ ]:
prompt = "Write a Python code generating a poem about Transformers."
generate_and_print_text(model, prompt, tokenizer, max_new_tokens=300, print_model_input=True)

# Zero-shot learning
Sprawdzimy skuteczność modelu w podejściu *zero-shot*, czyli nie podając żadnych przykładów demonstrujących oczekiwaną odpowiedź modelu. Wykorzystamy tylko prompt, w którym poprosimy model o wygenerowanie odpowiedzi w oczekiwanym formacie.

In [ ]:
def test_prompt_with_instruction(data_point) -> str:
    prompt =f"""Convert the following text to a JSON format.
You can only use the JSON keys: {list(attributes_to_keep)}. Not all keys are required.
###
{data_point["text"]}
###
"""
    return prompt

In [ ]:
data_point = test_dataset[1]
print(data_point)
print("\nPrompt:")
prompt = test_prompt_with_instruction(data_point)
print(prompt)

In [ ]:
model_input = tokenizer(prompt, return_tensors="pt").to(device)
print(model_input)


In [ ]:
generate_and_print_text(model, prompt, tokenizer, max_new_tokens=70)

Sprawdź, czy odpowiedź jest w formacie JSON. Model bazowy, który nie był dostrajany do wykonywania poleceń, często generuje zamiast tego inny tekst, np. dalszy ciąg polecenia albo kod w Pythonie.

# Zadanie 1 – Few-shot learning


W tej części notatnika zaimplementuj metodę *few-shot learning*, polegającą na dodaniu do promptu kilku przykładów demonstrujących oczekiwane działanie modelu, aby zwiększyć szanse wygenerowania poprawnej odpowiedzi.
Konstruując prompt z przykładami demonstrującymi oczekiwaną odpowiedź modelu, możesz wykorzystać polecenie użyte w prompcie w części "Zero-shot learning" notatnika.

Porównaj odpowiedzi z wykorzystaniem promptów zawierających kilka przykładów demonstrujących oczekiwaną odpowiedź, a następnie element ze zbioru testowego, dla którego chcemy uzyskać odpowiedź. Sprawdź odpowiedzi modelu z wykorzystaniem zera, jednego i pięciu przykładów demonstrujących oczekiwaną odpowiedź – za każdym razem dla tego samego elementu zbioru testowego.

In [ ]:
# Zero przykładów demonstrujących oczekiwaną odpowiedź (zero-shot learning)


In [ ]:
# Jeden przykład demonstrujący oczekiwaną odpowiedź (1-shot learning)


In [ ]:
# Pięć przykładów demonstrujących oczekiwaną odpowiedź (5-shot learning)


Oceń ilościowo metodę few-shot na zbiorze testowym funkcją `evaluate_json`. Dla wszystkich zapytań testowych wykorzystaj ten sam zestaw pięciu przykładów demonstracyjnych.

In [ ]:
few_shot_demonstration = ...  # TODO: prompt z pięcioma przykładami demonstracyjnymi
results["few-shot (5 przykładów)"] = ...  # TODO: evaluate_json(model, test_dataset, prompt_fn, n=N_EVAL)
raise NotImplementedError("TODO: zbuduj prompt few-shot i oceń go przez evaluate_json")

# Zadanie 2 – Wybrana metoda efektywnego dostrojenia modelu

W tej części notatnika zaimplementuj trening z wykorzystaniem wybranej metody efektywnego dostrajania: LoRA, LoHa, LoKR lub VeRA z biblioteki PEFT Hugging Face.
Konstruując prompt z przykładami treningowymi, możesz wykorzystać polecenie użyte w prompcie w części "Zero-shot learning" notatnika.

Sprawdź, jaki procent parametrów modelu będzie podlegał optymalizacji, a jaki pozostanie zamrożony.

Wytrenuj model, stosując zaimplementowaną metodę. Porównaj odpowiedzi z modelu bazowego i modelu dostrojonego na kilku przykładach ze zbioru testowego.

Wskazówki:
* nazwy warstw do parametru `target_modules` dobierz na podstawie nazw warstw liniowych modelu (komórka wypisująca je jest poniżej),
* przykłady treningowe tokenizuj funkcją `tokenize_completion_only`, a w klasie `Trainer` użyj `data_collator=transformers.default_data_collator`,
* do oceny w trakcie treningu użyj zbioru walidacyjnego `val_dataset` (nie testowego),
* na GPU T4 wystarczy ok. 300 kroków treningu (stała uczenia ok. 2e-4, ewaluacja co 100 kroków); w `FAST_MODE` wystarczy kilka kroków,
* po treningu zapisz adapter (`save_pretrained`) i oceń model funkcją `evaluate_json`.

Utworzenie instancji modelu przygotowanej do dostrojenia metodą LoRA.

**Uwaga:** `get_peft_model` modyfikuje przekazany model w miejscu – dodaje do jego warstw adaptery LoRA (kopia modelu, np. `copy.deepcopy(model)`, podwoiłaby zajętość pamięci). Odpowiedzi modelu bazowego uzyskamy, tymczasowo wyłączając adapter:
```python
with peft_model.disable_adapter():
    generate_and_print_text(peft_model, prompt, tokenizer)
```

Model skwantyzowany do 4 bitów przed dodaniem adapterów przygotowujemy funkcją `prepare_model_for_kbit_training` (zamrożenie wag, rzutowanie pozostałych, nieskwantyzowanych parametrów – normalizacji, embeddingów, `lm_head` – do fp32, włączenie *gradient checkpointing*). W FAST_MODE (CPU) model nie jest skwantyzowany i ten krok jest pomijany.

Nazwy warstw, do których dodajemy adaptery LoRA (parametr `target_modules`), zależą od implementacji modelu. Wypiszmy nazwy warstw liniowych (`nn.Linear`) w modelu Phi. W modelu skwantyzowanym są to warstwy `bitsandbytes` (`Linear4bit`), które dziedziczą po `nn.Linear`.

In [ ]:
linear_layer_names = sorted({name.split(".")[-1] for name, module in model.named_modules()
                             if isinstance(module, nn.Linear)})
print(linear_layer_names)

In [ ]:
config = ...  # TODO: LoraConfig(...)
peft_model = ...  # TODO: get_peft_model(model, config)
raise NotImplementedError("TODO: utwórz config (LoraConfig) i peft_model (get_peft_model)")

Sprawdzenie liczby dostrajanych parametrów w modelu trenowanym z wykorzystaniem metody LoRA.

In [ ]:
peft_model.print_trainable_parameters()

Dostrojenie modelu `peft_model` z wykorzystaniem klasy Trainer.

Zapisz adapter metodą `save_pretrained` (w osobnym katalogu dla każdej metody) i sprawdź, czy da się go ponownie wczytać do świeżo wczytanego modelu bazowego (`PeftModel.from_pretrained`). Pliki zapisane w Colabie znikają po zakończeniu sesji – aby zachować adapter, skopiuj katalog np. na Dysk Google.

Porównaj odpowiedzi bazowej, niedostrojonej wersji modelu oraz dostrojonej wersji modelu (`peft_model`) dla kilku przykładowych elementów z testowego zbioru danych. Odpowiedzi modelu bazowego uzyskasz w bloku `with peft_model.disable_adapter():`.

Oceń ilościowo model dostrojony wybraną metodą na zbiorze testowym. Do wyników możesz dopisać liczbę trenowanych parametrów (`peft_model.get_nb_trainable_parameters()[0]`) i czas treningu (`trainer.train().metrics["train_runtime"]`) – zostaną uwzględnione w tabeli porównawczej.

In [ ]:
results["LoRA"] = ...  # TODO: evaluate_json(peft_model, test_dataset, test_prompt, n=N_EVAL)

# Zadanie 3 – Wybrana metoda dostrojenia promptu

W tej części notatnika zaimplementuj trening z wykorzystaniem wybranej metody dostrajania promptu: prompt tuning, prefix tuning lub p-tuning z biblioteki PEFT Hugging Face.

Sprawdź, jaki procent parametrów będzie podlegał optymalizacji, a jaki pozostanie zamrożony.

Wytrenuj model, stosując zaimplementowaną metodę. Porównaj odpowiedzi z modelu bazowego i modelu dostrojonego na kilku przykładach ze zbioru testowego.

Wygeneruj ponownie zbiór treningowy i walidacyjny. Nie dodawaj polecenia "Convert the following text to a JSON format..." do promptów w zbiorze treningowym.
Zamiast dodawać polecenie w języku naturalnym, wykorzystamy wirtualny prompt optymalizowany w procesie uczenia.

Ponownie przygotuj zbiór treningowy i walidacyjny po tokenizacji, korzystając z nowego promptu.

Utworzenie instancji modelu przygotowanej do dostrojenia metodą *prompt tuning*. W początkowym prompcie podlegającym optymalizacji warto umieścić listę kluczy JSON w wynikowym tekście (zmienna `attributes_to_keep`). Przykładowy początkowy prompt mógłby wyglądać następująco:

```
initial_prompt = f"{attributes_to_keep} xxx placeholder for virtual prompt xxx"
```


**Uwaga:** Przed wczytaniem nowego modelu zwolnij pamięć zajmowaną przez model z części LoRA (`del ...`, `gc.collect()`, `torch.cuda.empty_cache()`). Do prompt tuningu wczytaj model bazowy bez kwantyzacji (`load_base_model(quantize=False)`). Metody dostrajania promptu nie zmieniają warstw modelu bazowego, więc kopia modelu nie jest potrzebna, a odpowiedzi modelu bazowego uzyskasz w bloku `with peft_model.disable_adapter():`.

In [ ]:
peft_model.print_trainable_parameters()

Dostrojenie modelu `peft_model` z wykorzystaniem klasy `Trainer`.

Odpowiedź modelu bazowego (wirtualny prompt tymczasowo wyłączony: `peft_model.disable_adapter()`).

Odpowiedź modelu `peft_model` z dostrojonym promptem.

Oceń ilościowo model z dostrojonym promptem na zbiorze testowym (zadanie opcjonalne).

In [ ]:
# (opcjonalnie) ocena ilościowa modelu z dostrojonym promptem
results["prompt tuning"] = ...  # TODO: evaluate_json(peft_model, test_dataset, test_prompt, n=N_EVAL)

# Porównanie metod

Tabela z wynikami ilościowymi metod zapisanymi w słowniku `results`: liczba trenowanych parametrów, czas treningu [s], odsetek poprawnych odpowiedzi JSON, precyzja, czułość i F1 na poziomie slotów oraz liczba ocenianych przykładów testowych.

In [ ]:
import pandas as pd

columns = ["trainable_params", "train_time_s", "valid_json", "precision", "recall", "f1", "n"]
table = pd.DataFrame.from_dict({name: r for name, r in results.items() if isinstance(r, dict)}, orient="index")
table.reindex(columns=columns).round(3)

**Co dalej?** Opcjonalny lab 3b ([`lab3b_sft_dpo_student.ipynb`](../lab3b/lab3b_sft_dpo_student.ipynb)) pokazuje, jak dostraja się dziś asystentów czatowych: szablony czatu, SFT z biblioteką `trl` i DPO na parach preferencji, na modelu instrukcyjnym Qwen2.5-0.5B i polskich poleceniach.